# Forge — Score any robotics dataset in 60 seconds

[![GitHub](https://img.shields.io/badge/github-arpitg1304%2Fforge-181717?logo=github)](https://github.com/arpitg1304/forge) &nbsp; [![PyPI](https://img.shields.io/pypi/v/forge-robotics?color=6c9fff)](https://pypi.org/project/forge-robotics/) &nbsp; ⭐ if this saves you a Friday afternoon

Researchers ship robotics datasets without ever scoring demonstration quality. **A meaningful chunk of "published" demos contain dead time, gripper chatter, jerky motion, or are pure static.** This notebook shows you how to find them — across any dataset format — in three cells.

**What you'll do:**
1. Pick a public LeRobot dataset from a dropdown.
2. Download + inspect + score every episode on 8 proprio-based quality metrics.
3. Visualize the worst-scoring episode and see *why* it scored poorly.

No GPU needed. Runs in ~60 seconds.

## 1. Install

In [ ]:
%pip install -q "forge-robotics[lerobot,hub,visualize]"

## 2. Pick a dataset

Three small public LeRobot datasets — each downloads in a few seconds.

| Dataset | Robot | Episodes | Why it's interesting |
|---|---|---|---|
| `lerobot/pusht` | 2D pusher | 206 | Tiny, classic Diffusion Policy benchmark |
| `lerobot/aloha_sim_transfer_cube_human` | Bimanual ALOHA | 50 | Teleoperated, multi-camera |
| `lerobot/jaco_play` | Jaco arm | 1085 | Real hardware, language-conditioned |

In [ ]:
import ipywidgets as widgets
from IPython.display import display

DATASETS = {
    "lerobot/pusht (small, fast)": "hf://lerobot/pusht",
    "lerobot/aloha_sim_transfer_cube_human (bimanual)": "hf://lerobot/aloha_sim_transfer_cube_human",
    "lerobot/jaco_play (real hardware)": "hf://lerobot/jaco_play",
}

dataset_picker = widgets.Dropdown(
    options=list(DATASETS.keys()),
    description="Dataset:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)
display(dataset_picker)
print("\nRun the next cell after picking a dataset — the rest of the notebook will use your selection.")

## 3. Download + inspect

`forge.inspect` auto-detects the format, finds cameras, infers FPS + gripper index, and tells you what's missing for downstream conversion.

In [ ]:
import forge
from forge.hub import download_dataset

selected = DATASETS[dataset_picker.value]
print(f"Selected: {selected}")
print("Downloading (cached after first run)...\n")

local_path = download_dataset(selected)
print(f"  → cached at: {local_path}\n")

info = forge.inspect(str(local_path))
print(info.summary())
print("\nCameras:", list(info.cameras.keys()) or "(none)")
print("Inferred FPS:", info.inferred_fps)
print("State dim:", len(info.observation_schema))

## 4. Score every episode

Forge computes 8 quality metrics per episode from proprioception alone — no video processing needed:

1. **Smoothness (LDLJ)** — log dimensionless jerk; how shaky is the motion?
2. **Dead actions** — fraction of frames where the policy did nothing
3. **Gripper chatter** — open/close oscillations
4. **Path length** — joint-space distance traveled
5. **Timestamp regularity** — jitter in dt
6. **Action saturation** — fraction of frames at action-space limits
7. **Static detection** — frames where nothing moves
8. **Action entropy** — diversity of commands

Composite score is 0–10. Anything below 6 is suspect.

In [ ]:
from forge.quality import QualityAnalyzer

analyzer = QualityAnalyzer(fps=info.inferred_fps or 30.0)
report = analyzer.analyze_dataset(str(local_path))

print(f"Dataset: {report.dataset_path}")
print(f"Episodes scored: {report.num_episodes}")
print(f"Overall score:   {report.overall_score:.2f} / 10")
print(f"\nSubscores:")
for k, v in report.subscores.items():
    print(f"  {k:25s} {v:.2f}")
if report.recommendations:
    print("\nRecommendations:")
    for r in report.recommendations:
        print(f"  • {r}")

### Score distribution

How tight is the dataset's quality? A long left tail = some demos drag down the policy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

scores = np.array(
    [ep.overall_score for ep in report.per_episode if ep.overall_score is not None]
)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(scores, bins=20, color="#6c9fff", edgecolor="#1a2540")
ax.axvline(scores.mean(), color="#ff7c5c", linestyle="--", lw=2, label=f"mean = {scores.mean():.2f}")
p10 = float(np.percentile(scores, 10))
ax.axvline(p10, color="#888", linestyle=":", lw=1.5, label=f"10th percentile = {p10:.2f}")
ax.set_xlabel("Episode quality score (0–10)")
ax.set_ylabel("# episodes")
ax.set_title(f"{selected} — quality distribution across {len(scores)} episodes")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

n_low = (scores < p10).sum()
print(
    f"\nThe bottom decile ({n_low} of {len(scores)} episodes, score < {p10:.2f}) "
    f"is what `forge filter --min-quality {p10:.1f}` would drop. "
    f"On a well-curated dataset that's a tight band; on a noisy one it's where the surprises live."
)

### The 5 worst episodes

In [ ]:
scored = [ep for ep in report.per_episode if ep.overall_score is not None]
worst = sorted(scored, key=lambda ep: ep.overall_score)[:5]

print(f"{'episode':<20} {'score':>6}  {'frames':>6}  flags")
print("-" * 70)
for ep in worst:
    flags = ", ".join(ep.flags) if ep.flags else "-"
    print(f"{ep.episode_id:<20} {ep.overall_score:>6.2f}  {ep.num_frames:>6}  {flags}")

## 5. Visualize the worst episode

Pick which of the worst episodes to drill into. We'll plot the joint-space trajectory + show *why* the quality score is low.

In [ ]:
worst_picker = widgets.Dropdown(
    options=[(f"{ep.episode_id}  (score={ep.overall_score:.2f})", ep.episode_id) for ep in worst],
    description="Episode:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)
display(worst_picker)
print("\nRun the next cell to visualize.")

In [ ]:
from forge.formats.registry import FormatRegistry
import numpy as np
import matplotlib.pyplot as plt

fmt = FormatRegistry.detect_format(local_path)
reader = FormatRegistry.get_reader(fmt)
# Iterate to find the matching episode (works for chunked formats too).
episode = next(
    ep for ep in reader.read_episodes(local_path) if ep.episode_id == worst_picker.value
)
frames = episode.load_frames()

states = np.array([f.state for f in frames if f.state is not None])
actions = np.array([f.action for f in frames if f.action is not None])

n_axes = (1 if states.size else 0) + (1 if actions.size else 0)
n_axes = max(n_axes, 1)
fig, axes = plt.subplots(n_axes, 1, figsize=(10, 2.2 * n_axes), sharex=True)
if n_axes == 1:
    axes = [axes]

ax_idx = 0
if states.size:
    for d in range(states.shape[1]):
        axes[ax_idx].plot(states[:, d], alpha=0.7, lw=1, label=f"dim {d}")
    axes[ax_idx].set_ylabel("state")
    axes[ax_idx].legend(ncol=min(states.shape[1], 8), fontsize=7, loc="upper right")
    axes[ax_idx].spines["top"].set_visible(False); axes[ax_idx].spines["right"].set_visible(False)
    ax_idx += 1

if actions.size:
    for d in range(actions.shape[1]):
        axes[ax_idx].plot(actions[:, d], alpha=0.7, lw=1, label=f"dim {d}")
    axes[ax_idx].set_ylabel("action")
    axes[ax_idx].legend(ncol=min(actions.shape[1], 8), fontsize=7, loc="upper right")
    axes[ax_idx].spines["top"].set_visible(False); axes[ax_idx].spines["right"].set_visible(False)
    ax_idx += 1

# Overlay dead-action ranges (if any) as red bands.
ep_quality = next((e for e in worst if e.episode_id == worst_picker.value), None)
if ep_quality and ep_quality.dead_ranges:
    for ax in axes:
        for start, end in ep_quality.dead_ranges:
            ax.axvspan(start, end, color="#ff7c5c", alpha=0.15, lw=0)

axes[-1].set_xlabel("frame")
fig.suptitle(
    f"Episode {worst_picker.value}  —  score: {ep_quality.overall_score:.2f}  "
    f"—  flags: {', '.join(ep_quality.flags) or 'none'}\n"
    f"(red bands = dead-action ranges)",
    fontsize=11,
)
plt.tight_layout()
plt.show()

print("\nDiagnostic summary:")
if ep_quality.dead_fraction is not None:
    print(f"  dead-action fraction: {ep_quality.dead_fraction:.1%}")
if ep_quality.gripper_chatter_rate is not None:
    print(f"  gripper chatter:      {ep_quality.gripper_chatter_rate:.2f} flips/sec")
if ep_quality.ldlj is not None:
    print(f"  LDLJ smoothness:      {ep_quality.ldlj:.2f}  (less negative = smoother)")
if ep_quality.overall_saturation is not None:
    print(f"  action saturation:    {ep_quality.overall_saturation:.1%}")

### Show three camera frames from the episode

Three frames spread across the trajectory (¼ · ½ · ¾) so you see the action regardless of when in the episode the manipulation happens. Each thumbnail's title shows its mean pixel value — useful sanity check (some sims have a dark background that genuinely looks near-black; the ALOHA sim averages ~38/255).

In [ ]:
import io
import numpy as np
from PIL import Image
from IPython.display import display, HTML

if frames and frames[0].images:
    cam_name = next(iter(frames[0].images))
    n = len(frames)
    indices = [n // 4, n // 2, (3 * n) // 4]

    # Build a single side-by-side strip via PIL — bypasses matplotlib so we
    # don't hit any colormap / vmin-vmax / channel-order surprises.
    pil_imgs = []
    diagnostics = []
    for i in indices:
        arr = np.asarray(frames[i].images[cam_name].load())

        # Normalize whatever the reader returned to uint8 H×W×3 RGB.
        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))  # CHW -> HWC
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, axis=-1)  # grayscale -> RGB
        if arr.dtype != np.uint8:
            if arr.dtype.kind == "f" and arr.max() <= 1.0:
                arr = (arr * 255).clip(0, 255).astype(np.uint8)
            else:
                arr = arr.clip(0, 255).astype(np.uint8)

        pil_imgs.append(Image.fromarray(arr))
        diagnostics.append(
            f"frame {i}: shape={arr.shape} dtype={arr.dtype} "
            f"min={int(arr.min())} max={int(arr.max())} mean={float(arr.mean()):.1f}"
        )

    # Concatenate horizontally with a small gap.
    h = max(im.height for im in pil_imgs)
    pil_imgs = [im.resize((int(im.width * h / im.height), h)) for im in pil_imgs]
    gap = 8
    total_w = sum(im.width for im in pil_imgs) + gap * (len(pil_imgs) - 1)
    strip = Image.new("RGB", (total_w, h), color=(20, 20, 28))
    x = 0
    for im in pil_imgs:
        strip.paste(im, (x, 0))
        x += im.width + gap

    # Render at a sensible size in the notebook.
    target_w = 720
    if strip.width != target_w:
        strip = strip.resize((target_w, int(strip.height * target_w / strip.width)))

    buf = io.BytesIO()
    strip.save(buf, format="PNG")
    display(HTML(
        f'<div style="text-align:center;font-family:sans-serif;font-size:11px;color:#aaa">'
        f'<b>camera/{cam_name}</b> — episode {worst_picker.value}'
        f'</div>'
    ))
    display(Image.open(io.BytesIO(buf.getvalue())))

    print("\nFrame diagnostics (raw, before PIL):")
    for line in diagnostics:
        print(f"  {line}")
else:
    print("(this dataset has no images)")

## What just happened

In ~60 seconds, you:

- Loaded a published robotics dataset *without knowing or caring about its on-disk format* (LeRobot v2 in this case, but Forge handles RLDS / Zarr / HDF5 / MCAP / Rosbag the same way).
- Got an objective quality score for every episode.
- Spotted the worst demos and saw the specific failure mode (dead time, jerk, chatter).

This is the **same pipeline** that powers `forge filter`, which lets you drop low-quality episodes before training a policy. Train on better data → train fewer epochs → measurable downstream wins.

## Next steps

```bash
# Local install (full feature set + MCAP support)
pip install "forge-robotics[all]"

# CLI versions of everything you just did:
forge inspect    hf://lerobot/pusht
forge quality    hf://lerobot/pusht
forge filter     hf://lerobot/pusht ./out --min-quality 6.0
forge convert    hf://lerobot/pusht ./out --format lerobot-v3
forge visualize  ./out --backend rerun
```

**Try this on your own data.** Forge handles RLDS / LeRobot v2-v3 / Zarr / HDF5 / MCAP / Rosbag.

- ⭐ **GitHub:** https://github.com/arpitg1304/forge
- 🐘 **Docs:** https://arpitg1304.github.io/forge/
- 💬 **Issues / requests:** https://github.com/arpitg1304/forge/issues